In [ ]:
# BOB Alpha (mu) — Evaluation on LEMON (native 1000 Hz)
# Sensorimotor mu / alpha (9–13 Hz), C3 Hjorth, resting-state LEMON epochs.
# Conditions saved: DEFAULT (pre-optimization Burg) + OPT (Bayesian-optimized, Opt_o)
# Metrics: MACE, PLV, CSD (circular SD), Acc. GT = matched zero-phase FIR on the same channel.
# Saves one model-tagged JSON holding raw per-timepoint errors, both conditions' metrics,
# per-subject SNR + ITPC, and the per-subject BOB param configs.
import numpy as np
import os, time, json, gc, glob
from scipy.signal import butter, hilbert, lfilter, lfilter_zi, filtfilt, welch, firwin
from scipy.stats import pearsonr, spearmanr
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
try:
    from skopt import gp_minimize
    from skopt.space import Integer, Real
except ImportError:
    import subprocess; subprocess.check_call(['pip', 'install', 'scikit-optimize', '-q'])
    from skopt import gp_minimize
    from skopt.space import Integer, Real
print("setup ok")


Mounted at /content/drive
setup ok


In [ ]:
# ---- CONFIG ----
# Input: the per-subject LEMON .npz cache (C3 Hjorth, 1000 Hz, 9–13 Hz band).
# Each file holds: epochs (n_epochs, n_samples), fs, n_epochs, cal_n, test_n, montage.
LOAD_DIR = '/content/drive/MyDrive/RAIN_IRC/LEMON'
OUTDIR   = '/content/drive/MyDrive/RAIN_IRC/result_alpha/BOB_lemon'
os.makedirs(OUTDIR, exist_ok=True)

# Resting epochs have no stimulus onset, so eval times are measured from the epoch
# start (PRE_SAMP = 0). These match the PEAP LEMON evaluation timepoints.
EVAL_TIMES_S = [1.5, 2.0, 2.5, 3.0, 3.5]
HORIZON_MS   = 0.0
GT_LOW, GT_HIGH = 9.0, 13.0           # fixed mu/alpha GT band (both conditions scored against this)
GT_FIR_NUMTAPS  = 231                  # 1000 Hz; matches the PEAP FIR-230 GT (231 taps, Hamming default)
AR_CONTEXT_DIV  = 2
ZERO_PHASE      = True   # NEW: complete the zero-phase filter with a BACKWARD pass over the AR-padded
                         # signal. forward (lfilter on raw) + backward = zero-phase, matching the GT filtfilt.
                         # Set False to reproduce the original causal single-pass (for A/B testing).
                         # Fully causal either way: the pad is AR-extrapolated from past samples only,
                         # so the backward pass never sees a real future sample.
ALPHA_SEARCH_LO, ALPHA_SEARCH_HI = 9.0, 13.0   # range for the per-subject spectral-peak search
N_BAYES_CALLS, N_INITIAL, SEED = 100, 20, 42

# Baseline (pre-optimization) Burg config — naive literature defaults, kept for default-vs-opt.
# Band moved to mu/alpha; hilbert_pad ~one alpha cycle at 1000 Hz.
DEFAULT = {'ar_order': 96, 'context_s': 1.5, 'filt_low': 9.0, 'filt_high': 13.0,
           'filt_order': 2, 'hilbert_pad': 120}

# Search space scaled for mu/alpha at 1000 Hz (~77–111 samp/cycle, vs the theta-at-300/600 Hz
# version this was ported from). AR order spans roughly 0.3–1.8 cycles of the narrowband signal;
# context_s (s) and hilbert_pad (samp) are in their original units; filter edges bracket 9–13 Hz.
# If the optimizer pins AR/pad against a bound, widen that bound and rerun.
SEARCH_SPACE = [
    Integer(48, 180, name='ar_order'),
    Real(0.8, 2.0,    name='context_s'),
    Real(9, 10.5,   name='filt_low'),
    Real(11, 13,  name='filt_high'),
    Integer(1, 4,     name='filt_order'),
    Integer(60, 180,  name='hilbert_pad'),
]

files = sorted(glob.glob(os.path.join(LOAD_DIR, '*.npz')))
assert files, f"no npz found in {LOAD_DIR}"
_d0 = np.load(files[0], allow_pickle=True)
FS = float(_d0['fs']); PRE_SAMP = 0; _d0.close()      # resting epochs: t=0 at epoch start
HORIZON_SAMP = int(HORIZON_MS * FS / 1000.0)
EVAL_TIMES_SAMP = [PRE_SAMP + int(t * FS) for t in EVAL_TIMES_S]
print(f"FS={FS:.0f} Hz | t=0 at sample {PRE_SAMP} | GT {GT_LOW}-{GT_HIGH} Hz FIR-{GT_FIR_NUMTAPS}")
print(f"eval samples {EVAL_TIMES_SAMP} | {len(files)} subject files")


FS=1000 Hz | t=0 at sample 0 | GT 9.0-13.0 Hz FIR-231
eval samples [1500, 2000, 2500, 3000, 3500] | 20 subject files


In [ ]:
# ---- ENGINE ----
def make_causal_filter(lo, hi, order):
    nyq = FS / 2.0
    lo = max(0.5, min(lo, nyq - 1)); hi = max(lo + 0.5, min(hi, nyq - 0.5))
    return butter(int(order), [lo / nyq, hi / nyq], btype='band')

def causal_filter_signal(sig1d, b, a):
    zi = lfilter_zi(b, a) * sig1d[0]
    out, _ = lfilter(b, a, sig1d, zi=zi)
    return out.astype(np.float32)

def burg_ar_coeffs(x, order):
    x = x.astype(np.float64); n = len(x); order = int(order)
    if n <= order + 1: return np.zeros(order)
    ef = x.copy(); eb = x.copy(); a = np.zeros(order)
    for m in range(order):
        ef_s, eb_s = ef[m + 1:n], eb[m:n - 1]; L = min(len(ef_s), len(eb_s))
        if L <= 0: break
        ef_s, eb_s = ef_s[:L].copy(), eb_s[:L].copy()
        num = -2.0 * np.dot(ef_s, eb_s); den = np.dot(ef_s, ef_s) + np.dot(eb_s, eb_s)
        if den < 1e-12: break
        k = np.clip(num / den, -0.9999, 0.9999)
        a_new = np.zeros(m + 1); a_new[m] = k
        if m > 0: a_new[:m] = a[:m] + k * a[m - 1::-1]
        a[:m + 1] = a_new
        ef[m + 1:m + 1 + L] = ef_s + k * eb_s; eb[m + 1:m + 1 + L] = eb_s + k * ef_s; n = m + 1 + L
    return -a

def ar_extrapolate(x, coeffs, n_steps):
    p = len(coeffs); coeffs = np.asarray(coeffs, np.float64)
    out = np.empty(len(x) + n_steps, np.float64); out[:len(x)] = x
    for i in range(len(x), len(x) + n_steps):
        out[i] = out[i - 1] if i < p else np.dot(coeffs, out[i - p:i][::-1])
    return out

def burg_predict_phase(csig, predict_at, horizon, ar_order, context_s, hpad, b, a):
    csamp = int(context_s * FS); pad = int(hpad)
    start = max(0, predict_at - csamp + 1)
    x = csig[start:predict_at + 1].astype(np.float64)
    ar_order = min(int(ar_order), len(x) // AR_CONTEXT_DIV)
    if len(x) < ar_order + 5 or np.std(x) < 1e-10: return 0.0, 1.0
    ext = ar_extrapolate(x, burg_ar_coeffs(x, ar_order), horizon + pad)
    # csig is the FORWARD half of a zero-phase filter (lfilter, single causal pass). Now that the
    # AR pad supplies look-ahead, run the BACKWARD half over the padded signal: forward+backward
    # = zero-phase, aligning BOB's carrier phase with the GT's filtfilt. FULLY CAUSAL -- the pad is
    # AR-extrapolated from past samples only, so the backward pass sees no real future sample.
    if ZERO_PHASE:
        zi_b = lfilter_zi(b, a) * ext[-1]      # warm-start so the pad-side transient settles before tgt
        rev, _ = lfilter(b, a, ext[::-1], zi=zi_b)
        ext = rev[::-1]
    tgt = len(x) - 1 + horizon
    if tgt >= len(ext): return 0.0, 1.0
    ph = np.angle(hilbert(ext)[tgt]); return np.sin(ph), np.cos(ph)

def compute_gt(raw_epoch, lo, hi):
    nyq = FS / 2.0
    lo = max(0.5, min(lo, nyq - 1)); hi = max(lo + 0.5, min(hi, nyq - 0.5))
    taps = firwin(GT_FIR_NUMTAPS, [lo, hi], pass_zero=False, fs=FS)
    ph = np.angle(hilbert(filtfilt(taps, [1.0], raw_epoch.astype(np.float64))))
    return np.sin(ph).astype(np.float32), np.cos(ph).astype(np.float32)

def mace_from_errors(e):
    return 90.0 if len(e) < 10 else float(np.degrees(np.mean(np.abs(e))))

def errors_to_metrics(e):
    if len(e) < 10:
        return {'mace': 90.0, 'plv': 0.0, 'csd': 180.0, 'acc': 0.0, 'bias': 0.0, 'n_samples': 0}
    mean_abs = float(np.mean(np.abs(e))); R = float(np.abs(np.mean(np.exp(1j * e))))
    return {'mace': float(np.degrees(mean_abs)), 'plv': R,
            'csd': float(np.degrees(np.sqrt(-2.0 * np.log(max(R, 1e-12))))),   # circular SD (deg)
            'acc': float((1.0 - mean_abs / np.pi) * 100.0),
            'bias': float(np.degrees(np.angle(np.mean(np.exp(1j * e))))),
            'n_samples': int(len(e))}

def collect_errors(raw, gs_l, gc_l, trials, ar, ctx, lo, hi, ordf, hpad):
    try: b, a = make_causal_filter(lo, hi, int(ordf))
    except Exception: return np.array([])
    L = raw.shape[1]; errs = []
    for tr in trials:
        cs = causal_filter_signal(raw[tr], b, a)
        for es in EVAL_TIMES_SAMP:
            g = es + HORIZON_SAMP
            if g >= L: continue
            s, c = burg_predict_phase(cs, es, HORIZON_SAMP, int(ar), ctx, int(hpad), b, a)
            gsv, gcv = gs_l[tr][g], gc_l[tr][g]
            errs.append(np.arctan2(s * gcv - c * gsv, c * gcv + s * gsv))
    return np.array(errs)

def collect_errors_per_tp(raw, gs_l, gc_l, trials, ar, ctx, lo, hi, ordf, hpad):
    try: b, a = make_causal_filter(lo, hi, int(ordf))
    except Exception: return {t: np.array([]) for t in EVAL_TIMES_S}
    L = raw.shape[1]; by_t = {t: [] for t in EVAL_TIMES_S}
    for tr in trials:
        cs = causal_filter_signal(raw[tr], b, a)
        for t, es in zip(EVAL_TIMES_S, EVAL_TIMES_SAMP):
            g = es + HORIZON_SAMP
            if g >= L: continue
            s, c = burg_predict_phase(cs, es, HORIZON_SAMP, int(ar), ctx, int(hpad), b, a)
            gsv, gcv = gs_l[tr][g], gc_l[tr][g]
            by_t[t].append(np.arctan2(s * gcv - c * gsv, c * gcv + s * gsv))
    return {t: np.array(v) for t, v in by_t.items()}

def run_per_tp(raw, gs_l, gc_l, trials, p):
    return collect_errors_per_tp(raw, gs_l, gc_l, trials, p['ar_order'], p['context_s'],
                                 p['filt_low'], p['filt_high'], p['filt_order'], p['hilbert_pad'])

def itpc_from_gt(gs_l, gc_l, trials, esamp):
    ph = np.array([np.arctan2(gs_l[tr][esamp], gc_l[tr][esamp]) for tr in trials])
    return float(np.abs(np.mean(np.exp(1j * ph))))

def alpha_snr_per_epoch(raw):
    # mean over epochs of (9–13 Hz power / 1–40 Hz power); same formula as the LEMON loader's mu-SNR.
    snrs = []
    for tr in range(raw.shape[0]):
        ep = raw[tr].astype(np.float64)
        psd = np.abs(np.fft.rfft(ep)) ** 2
        freqs = np.fft.rfftfreq(len(ep), 1.0 / FS)
        mu = psd[(freqs >= 9) & (freqs <= 13)].mean()
        broad = psd[(freqs >= 1) & (freqs <= 40)].mean()
        snrs.append(mu / broad if broad > 0 else 0.0)
    return float(np.mean(snrs))

def estimate_alpha_peak(raw, onset_samp):
    seg = raw[:, onset_samp:]
    if seg.shape[1] < 16: return np.nan
    nfft = int(2 ** np.ceil(np.log2(seg.shape[1] * 4))); psd = None
    for tr in range(seg.shape[0]):
        f, px = welch(seg[tr].astype(np.float64), fs=FS, nperseg=seg.shape[1], nfft=nfft, detrend='linear')
        psd = px if psd is None else psd + px
    psd /= seg.shape[0]; band = (f >= ALPHA_SEARCH_LO) & (f <= ALPHA_SEARCH_HI)
    return float(f[band][np.argmax(psd[band])]) if np.any(band) else np.nan


In [ ]:
# ---- RUN ----
print(f"{'Subject':<12}{'N':>5}{'defMACE':>8}{'optMACE':>8}{'optPLV':>7}{'optCSD':>7}{'optAcc':>7}{'SNR':>6}{'ITPC':>6}{'Peak':>6}")
print('-' * 80)
results = []
for fp in files:
    t0 = time.time(); d = np.load(fp, allow_pickle=True); name = os.path.basename(fp)[:-4]
    if float(d['fs']) != FS:
        print(f"{name}: fs mismatch, skipping"); d.close(); continue
    raw = d['epochs'].astype(np.float32); n_tr = raw.shape[0]
    ch_name = str(d['montage']) if 'montage' in d.files else 'C3-Hjorth'
    cal_n = int(d['cal_n'])
    cal = np.arange(cal_n); test = np.arange(cal_n, n_tr)     # chronological split (matches LEMON loader)
    alpha_snr = alpha_snr_per_epoch(raw)

    gs_l, gc_l = [], []
    for tr in range(n_tr):
        s, c = compute_gt(raw[tr], GT_LOW, GT_HIGH); gs_l.append(s); gc_l.append(c)

    # --- baseline (pre-optimization) on TEST ---
    dflt_tp = run_per_tp(raw, gs_l, gc_l, test, DEFAULT)
    dflt_err = np.concatenate([dflt_tp[t] for t in EVAL_TIMES_S]) if any(len(dflt_tp[t]) for t in EVAL_TIMES_S) else np.array([])
    dflt_m = errors_to_metrics(dflt_err); dflt_tp_m = {t: errors_to_metrics(dflt_tp[t]) for t in EVAL_TIMES_S}

    # --- Bayesian optimization on CAL ---
    def objective(params):
        ar, ctx, lo, hi, ordf, hpad = params
        if lo >= hi - 0.5: return 90.0
        arc = min(int(ar), int(ctx * FS) // AR_CONTEXT_DIV)
        if arc < 4: return 90.0
        try: make_causal_filter(lo, hi, int(ordf))
        except Exception: return 90.0
        return mace_from_errors(collect_errors(raw, gs_l, gc_l, cal, arc, ctx, lo, hi, int(ordf), int(hpad)))
    res = gp_minimize(objective, SEARCH_SPACE, n_calls=N_BAYES_CALLS,
                      n_initial_points=N_INITIAL, random_state=SEED, verbose=False)
    bp = {'ar_order': int(res.x[0]), 'context_s': float(res.x[1]), 'filt_low': float(res.x[2]),
          'filt_high': float(res.x[3]), 'filt_order': int(res.x[4]), 'hilbert_pad': int(res.x[5])}
    bp['ar_order'] = min(bp['ar_order'], int(bp['context_s'] * FS) // AR_CONTEXT_DIV)

    # --- optimized on TEST ---
    opt_tp = run_per_tp(raw, gs_l, gc_l, test, bp)
    opt_err = np.concatenate([opt_tp[t] for t in EVAL_TIMES_S]) if any(len(opt_tp[t]) for t in EVAL_TIMES_S) else np.array([])
    opt_m = errors_to_metrics(opt_err); opt_tp_m = {t: errors_to_metrics(opt_tp[t]) for t in EVAL_TIMES_S}

    itpc_tp = {t: (itpc_from_gt(gs_l, gc_l, test, es) if es < raw.shape[1] else float('nan'))
               for t, es in zip(EVAL_TIMES_S, EVAL_TIMES_SAMP)}
    itpc_mean = float(np.nanmean(list(itpc_tp.values()))); f_peak = estimate_alpha_peak(raw, PRE_SAMP)

    results.append({
        'name': name, 'best_ch': ch_name, 'n_trials': int(n_tr),
        'n_cal': int(len(cal)), 'n_test': int(len(test)), 'alpha_snr': alpha_snr,
        'alpha_itpc_per_timepoint': {str(t): itpc_tp[t] for t in EVAL_TIMES_S},
        'alpha_itpc_mean': itpc_mean,
        'alpha_peak_hz': float(f_peak) if np.isfinite(f_peak) else None,
        'best_params': bp, 'default_params': DEFAULT,
        'default_metrics_pooled': dflt_m,
        'default_metrics_per_timepoint': {str(t): dflt_tp_m[t] for t in EVAL_TIMES_S},
        'default_errors_per_timepoint_rad': {str(t): [float(x) for x in dflt_tp[t]] for t in EVAL_TIMES_S},
        'opt_metrics_pooled': opt_m,
        'opt_metrics_per_timepoint': {str(t): opt_tp_m[t] for t in EVAL_TIMES_S},
        'opt_errors_per_timepoint_rad': {str(t): [float(x) for x in opt_tp[t]] for t in EVAL_TIMES_S},
    })
    pk = f"{f_peak:.2f}" if np.isfinite(f_peak) else " n/a"
    print(f"{name:<12}{opt_m['n_samples']:>5}{dflt_m['mace']:>7.1f}\u00b0{opt_m['mace']:>7.1f}\u00b0{opt_m['plv']:>7.3f}"
          f"{opt_m['csd']:>6.1f}\u00b0{opt_m['acc']:>6.1f}%{alpha_snr:>6.2f}{itpc_mean:>6.3f}{pk:>6}  [{time.time()-t0:.0f}s]")
    d.close(); del raw, gs_l, gc_l; gc.collect()
print(f"\n{len(results)} subjects done")


Subject         N defMACE optMACE optPLV optCSD optAcc   SNR  ITPC  Peak
--------------------------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(180), 2.0, 9.0, 13.0, np.int64(1), np.int64(180)] before, using random point [np.int64(97), 1.5707582662292627, 9.515991869703296, 11.088739322912629, np.int64(3), np.int64(142)]
  warnings.warn(


sub-032311    405   40.7°   29.8°  0.772  41.2°  83.5%  3.60 0.093 10.96  [204s]


/usr/local/lib/python3.12/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(180), 2.0, 9.0, 13.0, np.int64(1), np.int64(180)] before, using random point [np.int64(99), 1.6971338992553306, 10.495408021302456, 11.08525088233928, np.int64(2), np.int64(104)]
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(180), 2.0, 9.0, 13.0, np.int64(1), np.int64(180)] before, using random point [np.int64(56), 1.7629880528818584, 9.782563471119005, 12.086315716665723, np.int64(2), np.int64(126)]
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(180), 2.0, 9.0, 13.0, np.int64(1), np.int64(180)] before, using random point [np.int64(76), 0.9887856107877853, 10.412381447313576, 12.187731942907872, np.int64(2), np.int64(124)]
  warnings.

sub-032317    410   44.9°   33.3°  0.733  45.2°  81.5%  3.60 0.083 10.10  [223s]
sub-032324    400   34.1°   26.0°  0.816  36.6°  85.5%  3.90 0.087 10.56  [234s]


/usr/local/lib/python3.12/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(180), 2.0, 9.0, 13.0, np.int64(1), np.int64(180)] before, using random point [np.int64(89), 1.8206216978572989, 9.26873578003991, 11.784141370923452, np.int64(2), np.int64(140)]
  warnings.warn(


... (15 outputs left out)

/usr/local/lib/python3.12/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(180), 2.0, 9.0, 13.0, np.int64(1), np.int64(180)] before, using random point [np.int64(57), 1.5375885615499554, 10.401353236181054, 12.765569820621701, np.int64(3), np.int64(129)]
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(180), 2.0, 9.0, 13.0, np.int64(1), np.int64(180)] before, using random point [np.int64(50), 1.417935993637903, 9.44039165197793, 11.65816473636545, np.int64(1), np.int64(100)]
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(180), 2.0, 9.0, 13.0, np.int64(1), np.int64(180)] before, using random point [np.int64(121), 1.0827600125102808, 9.666349031952056, 12.186855613936109, np.int64(3), np.int64(154)]
  warnings.wa

sub-032511    440   46.3°   22.7°  0.861  31.4°  87.4%  4.34 0.095 12.02  [239s]


/usr/local/lib/python3.12/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(180), 2.0, 9.0, 13.0, np.int64(1), np.int64(180)] before, using random point [np.int64(124), 1.2516054092271707, 9.602464912318405, 12.287617719395584, np.int64(2), np.int64(86)]
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(180), 2.0, 9.0, 13.0, np.int64(1), np.int64(180)] before, using random point [np.int64(148), 1.610258038427447, 10.442227348523357, 11.177202222286322, np.int64(4), np.int64(76)]
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(180), 2.0, 9.0, 13.0, np.int64(1), np.int64(180)] before, using random point [np.int64(136), 1.968915336317348, 9.81911008546632, 11.423199400717202, np.int64(2), np.int64(160)]
  warnings.wa

sub-032524    430   61.7°   37.0°  0.696  48.8°  79.4%  3.84 0.130 10.86  [225s]
sub-032526    405   51.0°   31.8°  0.760  42.4°  82.3%  3.35 0.086 10.16  [277s]

20 subjects done


In [ ]:
# ---- SUMMARY + SAVE ----
n = len(results)
dM = np.array([r['default_metrics_pooled']['mace'] for r in results])
oM = np.array([r['opt_metrics_pooled']['mace'] for r in results])
oP = np.array([r['opt_metrics_pooled']['plv'] for r in results])
oC = np.array([r['opt_metrics_pooled']['csd'] for r in results])
oA = np.array([r['opt_metrics_pooled']['acc'] for r in results])
dA = np.array([r['default_metrics_pooled']['acc'] for r in results])
SNR = np.array([r['alpha_snr'] for r in results]); IT = np.array([r['alpha_itpc_mean'] for r in results])

print('=' * 72); print('BOB ALPHA (mu) — DEFAULT vs OPTIMIZED (Opt_o)'); print('=' * 72)
print(f"  Default   MACE {dM.mean():.1f}\u00b0\u00b1{dM.std():.1f}   Acc {dA.mean():.1f}%")
print(f"  Optimized MACE {oM.mean():.1f}\u00b0\u00b1{oM.std():.1f}   PLV {oP.mean():.3f}   CSD {oC.mean():.1f}\u00b0   Acc {oA.mean():.1f}%")
gain = dM - oM
print(f"  Optimization gain: {gain.mean():+.1f}\u00b0 MACE  ({int(np.sum(gain > 0))}/{n} improved)")
print(f"  (n={n} unique participants -- independent samples; group/inferential stats valid)")

# paired default-vs-optimized significance (per-subject, N independent participants, two-sided)
from scipy.stats import wilcoxon, ttest_rel, rankdata
diff = dM - oM; da = oA - dA   # >0 means optimization improved (MACE down / Acc up)
group_stats = {'n_participants': int(n), 'paired_test': 'two-sided',
               'mean_mace_default': float(dM.mean()), 'mean_mace_opt': float(oM.mean()),
               'mean_mace_reduction': float(diff.mean()), 'median_mace_reduction': float(np.median(diff)),
               'n_improved': int(np.sum(diff > 0))}
print('\n--- Paired default vs optimized (per-subject MACE, test set) ---')
print(f"  mean dMACE {diff.mean():+.2f}\u00b0   median {np.median(diff):+.2f}\u00b0   ({int(np.sum(diff > 0))}/{n} improved)")
if n >= 3 and np.any(diff != 0):
    w_stat, w_p = wilcoxon(dM, oM)
    nz = diff[diff != 0]; rk = rankdata(np.abs(nz))
    rbc = float((rk[nz > 0].sum() - rk[nz < 0].sum()) / rk.sum())
    t_stat, t_p = ttest_rel(dM, oM)
    sd = diff.std(ddof=1); dz = float(diff.mean() / sd) if sd > 0 else float('nan')
    group_stats.update({'wilcoxon_W': float(w_stat), 'wilcoxon_p': float(w_p), 'rank_biserial_r': rbc,
                        'ttest_t': float(t_stat), 'ttest_p': float(t_p), 'cohen_dz': dz})
    print(f"  Wilcoxon signed-rank: W={w_stat:.1f}, p={w_p:.4g}  (rank-biserial r={rbc:+.3f})")
    print(f"  Paired t-test:        t={t_stat:+.2f}, p={t_p:.4g}  (Cohen's dz={dz:+.3f})")
    if np.any(da != 0):
        _, ap = wilcoxon(oA, dA); group_stats['accuracy_wilcoxon_p'] = float(ap)
        print(f"  Accuracy (opt-def):   median {np.median(da):+.2f} pts, Wilcoxon p={ap:.4g}")
else:
    group_stats['note'] = 'insufficient nonzero differences'
    print("  (insufficient nonzero differences for a paired test)")

print('\n--- Per-timepoint (default vs optimized) ---')
print(f"{'ms':>5}{'defMACE':>9}{'optMACE':>9}{'optPLV':>8}{'optCSD':>8}{'optAcc':>8}{'ITPC':>8}{'N':>7}")
for t in EVAL_TIMES_S:
    dm = np.array([r['default_metrics_per_timepoint'][str(t)]['mace'] for r in results])
    g = lambda k: np.array([r['opt_metrics_per_timepoint'][str(t)][k] for r in results])
    ti = np.array([r['alpha_itpc_per_timepoint'][str(t)] for r in results])
    print(f"{int(t * 1000):>5}{dm.mean():>8.1f}\u00b0{g('mace').mean():>8.1f}\u00b0{g('plv').mean():>8.3f}"
          f"{g('csd').mean():>7.1f}\u00b0{g('acc').mean():>7.1f}%{ti.mean():>8.3f}{g('n_samples').mean():>7.0f}")

print('\n--- Accuracy (optimized) correlations across subjects ---')
if np.std(SNR) > 0:
    r1, p1 = pearsonr(SNR, oA); rs1, ps1 = spearmanr(SNR, oA)
    print(f"  Acc vs SNR : r={r1:+.3f} (p={p1:.4f})  rho={rs1:+.3f} (p={ps1:.4f})")
if np.std(IT) > 0:
    r2, p2 = pearsonr(IT, oA); rs2, ps2 = spearmanr(IT, oA)
    print(f"  Acc vs ITPC: r={r2:+.3f} (p={p2:.4f})  rho={rs2:+.3f} (p={ps2:.4f})")

print('\n--- Per-subject optimal params ---')
print(f"{'Subject':<12}{'AR':>4}{'Ctx':>5}{'Low':>5}{'High':>6}{'Ord':>4}{'Pad':>5}{'Peak':>6}")
for r in results:
    p = r['best_params']; pk = r['alpha_peak_hz']
    print(f"{r['name']:<12}{p['ar_order']:>4}{p['context_s']:>5.2f}{p['filt_low']:>5.2f}"
          f"{p['filt_high']:>6.2f}{p['filt_order']:>4}{p['hilbert_pad']:>5}{(f'{pk:.2f}' if pk else 'n/a'):>6}")

out = {'model': 'BOB',
       'config': {'fs': FS, 'pre_samp': PRE_SAMP, 'eval_times_s': EVAL_TIMES_S, 'horizon_ms': HORIZON_MS,
                  'gt_band': [GT_LOW, GT_HIGH], 'gt_fir_numtaps': GT_FIR_NUMTAPS, 'n_bayes_calls': N_BAYES_CALLS,
                  'dataset': 'LEMON top-20 mu-SNR (C3 Hjorth)', 'band': 'mu/alpha 9-13 Hz',
                  'split': 'chronological cal/test (first cal_n epochs = cal)', 'conditions': ['default', 'opt'],
                  'itpc_basis': 'GT-phase ITPC, test trials, per eval point', 'errors_unit': 'radians'},
       'group_stats': group_stats,
       'subjects': results}
outpath = os.path.join(OUTDIR, 'bob_alpha_results.json')
with open(outpath, 'w') as f:
    json.dump(out, f, indent=2)
print(f"\nSaved -> {outpath}")
print("Each subject holds default + opt metrics, per-timepoint errors (rad), SNR, ITPC, and BOB params.")
print("Comparators (e.g. PEAP): emit the same schema with their own 'model' tag, then load all and join on 'name'.")


BOB ALPHA (mu) — DEFAULT vs OPTIMIZED (Opt_o)
  Default   MACE 43.5°±7.0   Acc 75.9%
  Optimized MACE 28.3°±5.1   PLV 0.793   CSD 38.8°   Acc 84.3%
  Optimization gain: +15.2° MACE  (20/20 improved)
  (n=20 unique participants -- independent samples; group/inferential stats valid)

--- Paired default vs optimized (per-subject MACE, test set) ---
  mean dMACE +15.19°   median +14.26°   (20/20 improved)
  Wilcoxon signed-rank: W=0.0, p=1.907e-06  (rank-biserial r=+1.000)
  Paired t-test:        t=+13.84, p=2.251e-11  (Cohen's dz=+3.094)
  Accuracy (opt-def):   median +7.92 pts, Wilcoxon p=1.907e-06

--- Per-timepoint (default vs optimized) ---
   ms  defMACE  optMACE  optPLV  optCSD  optAcc    ITPC      N
 1500    43.8°    29.5°   0.780   39.9°   83.6%   0.081     84
 2000    44.1°    29.0°   0.786   39.5°   83.9%   0.113     84
 2500    42.8°    27.1°   0.808   36.8°   85.0%   0.086     84
 3000    43.6°    28.1°   0.793   38.6°   84.4%   0.102     84
 3500    43.0°    27.8°   0.802   3